# 6-7 Store

In [ ]:
import pandas as pd
import numpy as np
import pytz
import pulp
import random
from datetime import datetime
from tqdm import tqdm

import plotly.graph_objects as go
from plotly.subplots import make_subplots


import warnings
warnings.filterwarnings("ignore")

In [6]:
df_complete = pd.read_parquet('raw_data/df_sales_country.parquet', engine='pyarrow')

display(df_complete.head(5))
display(df_complete[['net_sales', 'gross_sales', 'qty_sold']].describe())

,dt_ref,store_id,state,sales_region,net_sales,gross_sales,qty_sold
0,2026-01-01,STORE-0067,New York,Northeast,36361.16,42056.28,4019
1,2026-01-01,STORE-0074,New York,Northeast,54537.80,66148.37,4312
2,2026-01-01,STORE-0264,Michigan,Midwest,6662.84,7734.35,567
3,2026-01-01,STORE-0307,Connecticut,Northeast,15012.83,19541.29,2383
4,2026-01-01,STORE-0384,New York,Northeast,36296.36,44760.88,3033


,net_sales,gross_sales,qty_sold
count,4.402570e+05,4.402570e+05,440257.0
mean,2.223055e+04,2.658801e+04,2820.979078
std,2.535722e+04,2.959463e+04,4454.221994
min,6.300000e-01,6.900000e-01,1.0
25%,1.065993e+04,1.278014e+04,1359.0
50%,1.596257e+04,1.910286e+04,2031.0
75%,2.550842e+04,3.052686e+04,3178.0
max,2.827760e+06,2.884773e+06,679834.0


In [14]:
def stores_optimizer(df_state: pd.DataFrame, criteria: str):
    df = df_state.copy()
    # 1. IDENTIFICAÇÃO DOS GRUPOS
    # Lojas que pertencem ao grupo de teste
    list_store_test = (df.loc[df['flag_test_control'] == 'test', 'store_id'].dropna().unique().tolist())
    df_store_test = pd.DataFrame({'store_id': list_store_test, 'flag': 'test'})

    # Identificação dos grupos
    df_temp_opt = df.merge(df_store_test, on='store_id', how='left')
    df_temp_opt['flag'] = df_temp_opt['flag'].fillna('control')

    # 2. CONVERSÃO DOS DADOS
    # Lojas candidatas ao grupo de controle
    store_set = (df_temp_opt.loc[df_temp_opt['flag'] == 'control', 'store_id'].dropna().unique().tolist())

    # Quantidade de lojas de teste
    n_test_stores = len(list_store_test)

    # Vendas diárias do grupo de teste
    daily_sales = (df_temp_opt.loc[df_temp_opt['flag'] == 'test'].groupby('dt_ref')[criteria].sum().to_dict())

    # Vendas diárias por loja candidata ao controle
    sales_store_id = (df_temp_opt.loc[df_temp_opt['flag'] == 'control'].groupby(['store_id', 'dt_ref'])[criteria].sum().to_dict())


    # 3. DECLARAÇÃO DO PROBLEMA
    prob = pulp.LpProblem('experiment_design', pulp.LpMinimize)

    # Variáveis binárias de seleção das lojas
    x = prob.add_variable_dicts('store_id', store_set, lowBound=0, upBound=1, cat=pulp.LpInteger)

    # Diferença percentual absoluta por dia
    sales_dif = prob.add_variable_dicts('sales_dif', list(daily_sales.keys()), lowBound=0, cat=pulp.LpContinuous)


    # 4. RESTRIÇÕES
    for day in daily_sales:
        sales_control = pulp.lpSum(x[store] * sales_store_id.get((store, day), 0) for store in store_set)

        # Normalização pelo faturamento do grupo teste
        denominator = max(abs(daily_sales[day]), 1)
        difference = (sales_control - daily_sales[day]) / denominator
        # Valor absoluto da diferença percentual
        prob += sales_dif[day] >= difference
        prob += sales_dif[day] >= -difference


    # 5. FUNÇÃO OBJETIVO
    prob += pulp.lpSum(sales_dif[day] for day in daily_sales)

    # 6. RESTRIÇÃO DE QUANTIDADE DE LOJAS
    prob += (pulp.lpSum(x[store] for store in store_set) == n_test_stores)

    # 6. RESTRIÇÕES
    # C1: Minimizar a diferença absoluta das vendas diárias
    for day in daily_sales:
        sales_control = pulp.lpSum(x[store] * sales_store_id.get((store, day), 0) for store in store_set)
        prob += (sales_dif[day] >= sales_control - daily_sales[day])
        prob += (sales_dif[day] >= daily_sales[day] - sales_control)

    # C2: Igualar a quantidade de lojas dos grupos
    prob += (pulp.lpSum(x[store] for store in store_set) == n_test_stores)


    # 7. RESOLUÇÃO
    solver = pulp.COIN_CMD(msg=1, timeLimit=EXECUTION_TIME, gapRel=0.001)
    stats = prob.solve(solver)

    # 8. EXTRAÇÃO DOS RESULTADOS

    if not stats.has_solution:
        raise RuntimeError(f'O modelo não encontrou uma solução. Status: {stats.status_str}')

    control_group_store = [store for store in store_set if x[store].value() is not None and x[store].value() > 0.5]
    control_group = pd.DataFrame({
        'store_id': control_group_store,
        'flag': 'control'
    })

    return pd.concat([control_group, df_store_test])


In [15]:
def create_test_group(df):
    test_stores = []

    all_stores = df['store_id'].unique()
    store_count = int(len(all_stores) * PERCENT_TEST_STORES)
    temp_test_stores = random.sample(sorted(all_stores), store_count)  
    temp_test_stores = pd.DataFrame(temp_test_stores, columns=['store_id'])
    temp_test_stores['flag_test_control'] = 'test'
    test_stores.append(temp_test_stores)

    if len(test_stores) == 0:
        return None

    test_stores = pd.concat(test_stores)

    df = pd.merge(df, test_stores, how='left', on='store_id')
    return df   

In [16]:
EXECUTION_TIME = 120
PERCENT_TEST_STORES = 0.25
CRITERIA= 'qty_sold' #net_sales, gross_sales

dt_carga = datetime.now(tz = pytz.timezone('America/Chicago'))
lst_df_result = []

#states = sorted(df_complete['state'].unique().tolist())

states = ['New York']

for state in tqdm(states):
    df_state = df_complete.loc[df_complete['state'] == state]

    test_id = str(state) + '-' + str(dt_carga).split(' ')[0]


    # Creating test group
    df_state = create_test_group(df_state)

    if df_state is not None:
        # Running the optimization algorithm
        lst_df_result.append(stores_optimizer(df_state, criteria=CRITERIA))
        

df_result = pd.concat(lst_df_result)

df_result = pd.merge(df_complete, df_result, how='left', on=['store_id'])
    

100%|██████████| 1/1 [02:00<00:00, 120.48s/it]

Welcome to the CBC MILP Solver
Version: Devel (unstable)
Build Date: Mar 23 2026
command line - C:\Users\krupc\AppData\Local\Temp\d8eb75152e8a4c29bfb8a2eb36e375ef-pulp.mps -sec 120 -ratio 0.001 -timeMode elapsed -solve -printingOptions all -solution C:\Users\krupc\AppData\Local\Temp\d8eb75152e8a4c29bfb8a2eb36e375ef-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1291 COLUMNS
At line 166885 RHS
At line 168172 BOUNDS
At line 168305 ENDATA
Problem MODEL has 1286 rows, 453 columns and 165008 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+08 to 120
ratioGap was changed from 0 to 0.001
Continuous objective value is 1.27384e+06 - 0.155 seconds
Cgl0004I processed model has 1285 rows, 453 columns (132 integer (132 of which binary)) and 164876 elements
Coin0011I Conflict graph built in 0.001 seconds, density: 0.032%
Cgl0015I Clique Strengthening extended 0 cliques, 0 were dominated
Cbc0045I Nauty sparseSpace 990264 affine 164098 coeffic

In [10]:
df_result

,dt_ref,store_id,state,sales_region,net_sales,gross_sales,qty_sold,flag
0,2026-01-01,STORE-0067,New York,Northeast,36361.16,42056.28,4019,NaN
1,2026-01-01,STORE-0074,New York,Northeast,54537.80,66148.37,4312,NaN
2,2026-01-01,STORE-0264,Michigan,Midwest,6662.84,7734.35,567,NaN
3,2026-01-01,STORE-0307,Connecticut,Northeast,15012.83,19541.29,2383,NaN
4,2026-01-01,STORE-0384,New York,Northeast,36296.36,44760.88,3033,test
...,...,...,...,...,...,...,...,...
440252,2026-11-17,STORE-1444,Michigan,Midwest,13886.49,16733.44,1969,NaN
440253,2026-11-17,STORE-1445,Texas,South,7268.93,9034.00,967,NaN
440254,2026-11-17,STORE-1446,Illinois,Midwest,8844.89,10099.56,799,NaN
440255,2026-11-17,STORE-1447,Washington,West,7451.21,8422.18,620,NaN


In [20]:


df_plot = df_result.groupby(['dt_ref', 'flag'], as_index=False)[['net_sales', 'gross_sales', 'qty_sold']].sum()

# 1. PREPARAÇÃO DOS DADOS
df_plot = (df_result.groupby(['dt_ref', 'flag'], as_index=False)['qty_sold'].sum())
df_plot['dt_ref'] = pd.to_datetime(df_plot['dt_ref'])

# Transformando as flags em colunas
df_pivot = (df_plot.pivot(index='dt_ref', columns='flag', values='qty_sold').fillna(0).sort_index())

# Diferença entre os grupos
df_pivot['difference'] = df_pivot['test'] - df_pivot['control']
df_pivot['difference_pct'] = ((df_pivot['test'] / df_pivot['control'].replace(0, float('nan'))) - 1) * 100

# 2. CRIAÇÃO DOS SUBPLOTS
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=('Quantidade Vendida - Test vs Control', 'Diferença de Quantidade - Test - Control'), row_heights=[0.65, 0.35])

# 3. GRÁFICO DE SÉRIE TEMPORAL
fig.add_trace(go.Scatter(x=df_pivot.index,y=df_pivot['test'],mode='lines',name='Test',line=dict(color='#2196F3', width=2)),row=1,col=1)
fig.add_trace(go.Scatter(x=df_pivot.index,y=df_pivot['control'],mode='lines',name='Control',line=dict(color='#FF9800', width=2)), row=1, col=1)

# 4. GRÁFICO DE DIFERENÇA
fig.add_trace(go.Bar(x=df_pivot.index, y=df_pivot['difference'], name='Test - Control', marker=dict(color=['#2196F3' if value >= 0 else '#EF5350' for value in df_pivot['difference']])), row=2, col=1)

# Linha de referência em zero
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1, row=2, col=1)

# 5. LAYOUT
fig.update_layout(
    title=dict(
        text='Comparativo de Quantidade Vendida - Test vs Control',
        x=0.5
    ),
    template='plotly_white',
    height=750,
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    barmode='relative'
)
fig.update_yaxes(title_text='Quantidade Vendida',row=1,col=1)
fig.update_yaxes(title_text='Diferença (Test - Control)',row=2,col=1)
fig.update_xaxes(title_text='Data',row=2,col=1)
fig.show()


In [21]:
df_pivot['difference_pct'].mean()

np.float64(0.2251687123857589)

# Introdução

# Problemática

# Solução

# To do
- [ ] Dá para utilizar métricas de machine learning?
- [ ] Principais gráficos a serem utilizados na apresentação: (Séries temporais, barra de diferença e mapa de calor de correlação);
- [ ] Escrever os textos de:
  - [ ] Introdução: Sobre o que é o projeto;
  - [ ] Problemática: O porque esse projeto é importante, principalmente em um contexto de testes de lojas físicas;
  - [ ] Solução: Algoritmos utilizados e lógica desenvolvida;
- [ ] Fazer em dois arquivos (Algoritmo e Resultados);
- [ ] Aplicar identidade visual semelhante a lojas 7-11;
- [ ] Traduzir para o inglês;
- [ ] Colocar no portfólio;